# EDA — BANKING77 (train)

Cel: poznać dane, z którymi pracujemy — rozkład klas, długości, duplikaty, niejednoznaczne intencje — i zapisać wnioski, które pójdą do README (raport) oraz do decyzji o modelach (`class_weight`, `max_len`).

**Zasady:**
- tylko `train.csv` przez `load_train()`; test jest zamknięty do oceny końcowej w W3 — w tym notebooku **nigdy** `load_test()` (strażnik nie skanuje notebooków, tę granicę pilnujemy sami);
- nie edytujemy plików CSV (suma SHA-256);
- przed commitem: *Kernel → Restart Kernel and Clear Outputs of All Cells* — w repo bez surowych tekstów poza pojedynczymi przykładami.

Szkielet: asystent (Claude), 5.10.2026 · analizy i wnioski: Marcin.

## 0. Wczytanie danych

In [ ]:
import json

import pandas as pd

from banking77_intent_ml.config import load_settings
from banking77_intent_ml.data.labels import load_label_map
from banking77_intent_ml.data.loader import load_train
from banking77_intent_ml.data.splits import VAL_IDS_FILE, normalize_text  # noqa: F401 (sekcja 5)

pd.set_option("display.max_colwidth", 200)  # pełne teksty wiadomości w tabelach

settings = load_settings()
train = load_train(settings)  # kolumny: text, category; indeks = pozycja wiersza w train.csv
label_map = load_label_map(settings.dataset.label_map_path)  # 77 kategorii -> id
val_ids = json.loads((settings.paths.splits_dir / VAL_IDS_FILE).read_text(encoding="utf-8"))
is_val = train.index.isin(val_ids)  # True = wiersz w części walidacyjnej zamrożonego splitu

print(train.shape, "| kategorii:", train["category"].nunique(), "| w walidacji:", int(is_val.sum()))
train.head()

## 1. Ogólny obraz i braki

Kształt, kolumny, typy. Braki: loader gwarantuje brak pustych tekstów i kategorii (inaczej dane by się nie wczytały). Rubryka wymaga „braki i ich powody” — tu: zero braków + granica wiedzy (autorzy zbioru nie dokumentują czyszczenia; README → *Dane*).

In [ ]:
train.info()
train.describe(include="all")


**Wniosek (Marcin):**
- 10 003 wiadomości, 77 kategorii.
- Brak pustych tekstów i etykiet; obie kolumny tekstowe.
- Brak informacji o wcześniejszym czyszczeniu danych.
- *(dopisek asystenta)* Brak braków to gwarancja, nie przypadek: loader przerywa wczytanie przy pustym tekście albo kategorii. Konsekwencja: bez imputacji i bez usuwania wierszy, więc pozycje wierszy, na których opiera się split, się nie przesuwają.
- *(dopisek asystenta)* `unique` dla `text` = 10 003 na 10 003, czyli zero dosłownych duplikatów. Wszystkie 31 duplikatów z sekcji 5 różnią się tylko wielkością liter, interpunkcją albo odstępami (np. „top up” i „top-up”), dlatego split grupuje po tekście znormalizowanym, a nie surowym.

## 2. Rozkład klas

Train jest niezrównoważony (35–187 na klasę), test ma dokładnie 40 na klasę (wiemy z audytu integralności, nie z oglądania testu). Pytania: które klasy są najmniej liczne? Czy niezrównoważenie może zaszkodzić macro-F1 na walidacji? (decyzja o `class_weight` — przed N1).

In [ ]:
import matplotlib.pyplot as plt

counts = train["category"].value_counts()  # liczność klas, malejąco
print(
    f"klas: {counts.size} | min {counts.min()} | mediana {counts.median():.0f} "
    f"| max {counts.max()} | max/min = {counts.max() / counts.min():.1f}"
)
print("5 najmniej licznych:", counts.tail(5).to_dict())

ax = counts.sort_values().plot.barh(figsize=(8, 16), title="Liczność klas w train.csv")
ax.axvline(counts.median(), color="gray", linestyle="--", label="mediana")
ax.set_xlabel("liczba przykładów")
ax.legend()
plt.tight_layout()
plt.show()

# Podział z manifestu: ile przykładów każda klasa ma w train i w walidacji
part = pd.Series(is_val, index=train.index, name="część").map({False: "train", True: "val"})
parts = pd.crosstab(train["category"], part)
parts["udział_val"] = (parts["val"] / (parts["train"] + parts["val"])).round(3)
share = parts["udział_val"]
print(f"udział walidacji w klasie: min {share.min()} | max {share.max()}")
parts.sort_values("val").head(10)  # klasy z najmniejszą liczbą przykładów walidacyjnych


**Wniosek (Marcin):**
- 35–187 przykładów na klasę; mediana 127, różnica ok. 5,3 razy.
- Najmniej danych: contactless_not_working i virtual_card_not_working.
- Walidacja ok. 15% w każdej klasie; dla najmniejszej tylko 5 przykładów.
- Macro-F1 i wyniki per klasa; class_weight do porównania na walidacji.


## 3. Przykłady i rodziny podobnych intencji

Losowe teksty z kilku kategorii, potem rodziny po początku nazwy kategorii: `card_*`, `top_up_*`, `pending_*`, `transfer_*`, `*_not_working` — tam spodziewamy się pomyłek modeli. Z podglądu: „Give me my card, I need it now!” ma etykietę `card_delivery_estimate`, choć pasowałby też do `card_arrival`.

In [ ]:
from IPython.display import display

selected_categories = [
    "card_arrival", "card_delivery_estimate",
    "pending_top_up", "top_up_reverted",
    "pending_transfer", "transfer_timing",
    "card_not_working", "contactless_not_working",
]

samples = (
    train.loc[train["category"].isin(selected_categories)]
    .groupby("category")
    .sample(n=2, random_state=settings.split.random_state)
    .sort_values("category")
)
display(samples)

families = {
    "card_*": [name for name in label_map if name.startswith("card_")],
    "top_up_*": [name for name in label_map if name.startswith("top_up_")],
    "pending_*": [name for name in label_map if name.startswith("pending_")],
    "transfer_*": [name for name in label_map if name.startswith("transfer_")],
    "*_not_working": [name for name in label_map if name.endswith("_not_working")],
}
family_summary = pd.DataFrame([
    {"rodzina": name, "liczba klas": len(categories), "kategorie": categories}
    for name, categories in families.items()
])
display(family_summary)


**Wniosek (Marcin):**
- Podobne pytania w card_arrival i card_delivery_estimate.
- Wspólny temat, różny stan operacji: pending_top_up i top_up_reverted.
- Ryzyko pomyłek między bliskimi intencjami; znaczenie szczegółów i kontekstu.


## 4. Długości tekstów (znaki, słowa, tokeny)

Znaki, słowa i tokeny DistilBERT z tokenami specjalnymi, bez obcinania. Rewizja tokenizera i `max_len` z configu. Pomiar tokenów powtórzony lokalnie poza notebookiem 12.10.2026 — wyniki i decyzja poniżej.

In [ ]:
lengths = pd.DataFrame({
    "znaki": train["text"].str.len(),
    "słowa": train["text"].str.split().str.len(),
})
display(lengths.describe(percentiles=[0.5, 0.95, 0.99]).round(1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for column, ax in zip(lengths.columns, axes, strict=True):
    lengths[column].plot.hist(bins=40, ax=ax, title=f"Długość tekstu: {column}")
    ax.axvline(lengths[column].median(), color="gray", linestyle="--", label="mediana")
    ax.set_xlabel(column)
    ax.set_ylabel("liczba wiadomości")
    ax.legend()
plt.tight_layout()
plt.show()

print("Najkrótsze teksty (z remisami):")
display(train.join(lengths).loc[lengths.nsmallest(5, "znaki", keep="all").index])
print("Najdłuższe teksty (z remisami):")
display(train.join(lengths).loc[lengths.nlargest(5, "znaki", keep="all").index])


In [ ]:
distilbert = settings.models.distilbert
token_lengths = None

if distilbert.revision == "PIN-ME":
    print("Tokeny: do policzenia po przypięciu rewizji DistilBERT w configu.")
else:
    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(
        distilbert.model_id, revision=distilbert.revision
    )
    encoded = tokenizer(
        train["text"].tolist(),
        add_special_tokens=True,
        truncation=False,
        padding=False,
        return_length=True,
    )
    token_lengths = pd.Series(encoded["length"], index=train.index, name="tokeny")
    display(token_lengths.describe(percentiles=[0.95, 0.99]).round(1))
    over_limit = token_lengths.gt(distilbert.max_len)
    print(
        f"Ponad max_len={distilbert.max_len}: {over_limit.sum()} wiadomości "
        f"({over_limit.mean():.2%}); długości z tokenami specjalnymi."
    )
    token_lengths.plot.hist(bins=40, figsize=(8, 4), title="Długość w tokenach DistilBERT")
    plt.axvline(distilbert.max_len, color="red", linestyle="--", label="max_len")
    plt.xlabel("liczba tokenów")
    plt.ylabel("liczba wiadomości")
    plt.legend()
    plt.tight_layout()
    plt.show()


**Wniosek (Marcin):**
- Krótkie wiadomości; mediana 47 znaków i 10 słów.
- 95% tekstów do 29 słów; maksimum 433 znaki i 79 słów.
- *(dopisek asystenta, pomiar 12.10.2026)* Tokenizer DistilBERT z przypiętej rewizji w configu, z tokenami specjalnymi i bez obcinania: mediana 13, P95 = 37, P99 = 53, maksimum 98 tokenów na 10 003 wiadomościach train.
- *(decyzja Marcina, 12.10.2026)* `max_len` zostaje 128 jak w configu: 0 wiadomości train przekracza limit, więc nie obcinamy żadnej z nich. Pomiar dotyczy wyłącznie train; nie przesądza o długościach przyszłych zgłoszeń.


## 5. Duplikaty po normalizacji i konflikt etykiet

Normalizacja z configu — `normalize_text` (NFKC → casefold → tokeny `\w+`), ta sama, której używa split. Z manifestu: 9 972 grupy (31 nadmiarowych duplikatów); grupa konfliktowa: wiersze 2002 i 3160 (`pending_top_up` / `top_up_reverted`) — w train, etykiet nie poprawiamy.

In [ ]:
audit = train.assign(
    klucz=train["text"].map(normalize_text),
    wiersz=train.index,
    część=part,
)
groups = audit.groupby("klucz").agg(
    liczba=("text", "size"),
    liczba_etykiet=("category", "nunique"),
    etykiety=("category", lambda values: sorted(set(values))),
    pozycje=("wiersz", list),
    liczba_części=("część", "nunique"),
)
duplicate_groups = groups.loc[groups["liczba"].gt(1)]
conflict_groups = groups.loc[groups["liczba_etykiet"].gt(1)]

print(
    f"Grupy: {len(groups)} | nadmiarowe duplikaty: {len(train) - len(groups)} | "
    f"grupy powtórzone: {len(duplicate_groups)} | "
    f"wiersze w tych grupach: {duplicate_groups['liczba'].sum()}"
)
display(duplicate_groups.drop(columns="liczba_etykiet").reset_index(drop=True))

assert duplicate_groups["liczba_części"].eq(1).all(), "Duplikaty po obu stronach splitu"
print("Duplikaty razem w jednej części splitu.")

conflict_positions = [position for rows in conflict_groups["pozycje"] for position in rows]
print(f"Grupy z konfliktem etykiet: {len(conflict_groups)}")
display(audit.loc[conflict_positions, ["text", "category", "część"]])


**Wniosek (Marcin):**
- 9 972 grupy po normalizacji; 31 par duplikatów, 62 wiersze.
- Jeden konflikt: pending_top_up i top_up_reverted, wiersze 2002 i 3160.
- Duplikaty razem w zamrożonym splicie; konflikt w części treningowej.
- Etykiety bez zmian.


## 6. Teksty skrajne i niejednoznaczne (10–20 z train)

Najkrótsze, najdłuższe i takie, przy których sam masz wątpliwość co do etykiety. Ying & Thomas (2022) wskazują automatycznie ~14% potencjalnych problemów etykiet w train — nie poprawiamy ich, opisujemy.

In [ ]:
short_ids = lengths.nsmallest(5, "znaki", keep="all").index
long_ids = lengths.nlargest(5, "znaki", keep="all").index
ambiguous = pd.Series({
    1034: "pilna prośba o kartę; card_arrival czy card_delivery_estimate?",
    2002: "ten sam tekst po normalizacji; pending_top_up",
    3160: "ten sam tekst po normalizacji; top_up_reverted",
    4046: "pytanie o PIN; etykieta get_physical_card",
    3917: "opis weryfikacji; forma odpowiedzi zamiast pytania klienta",
}, name="powód")

reasons = pd.concat([
    pd.Series("krótki tekst", index=short_ids, name="powód"),
    pd.Series("długi tekst", index=long_ids, name="powód"),
    ambiguous,
]).groupby(level=0, sort=False).agg("; ".join)

examples = train.join(lengths).loc[reasons.index].assign(powód=reasons)
print(f"Wybrane przykłady z train: {len(examples)}")
display(examples)


**Wniosek (Marcin):**
- Najkrótsze teksty z małą ilością kontekstu.
- Wśród najdłuższych opisy procedury weryfikacji, także w formie odpowiedzi.
- Niejednoznaczne etykiety i nakładające się intencje; materiał do późniejszej analizy błędów.


## 7. Słownictwo (zapowiedź TF-IDF)

Najczęstsze słowa w kilku klasach, zwłaszcza w rodzinach podobnych intencji z sekcji 3 — czy słowa je rozróżniają?

In [ ]:
from collections import Counter

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

word_rows = []
for category in selected_categories:
    texts = train.loc[train["category"].eq(category), "text"]
    words = Counter(
        word
        for text in texts
        for word in normalize_text(text).split()
        if word.isalpha() and len(word) > 1 and word not in ENGLISH_STOP_WORDS
    )
    for word, count in words.most_common(10):
        word_rows.append({"category": category, "słowo": word, "liczba": count})

vocabulary = pd.DataFrame(word_rows)
vocabulary["miejsce"] = vocabulary.groupby("category").cumcount() + 1
ranking = vocabulary.assign(
    słowo_liczba=vocabulary["słowo"] + " (" + vocabulary["liczba"].astype(str) + ")"
).pivot(index="miejsce", columns="category", values="słowo_liczba")
display(ranking[selected_categories])


**Wniosek (Marcin):**
- Card, transfer i money wspólne dla wielu intencji.
- Bardziej charakterystyczne słowa: track, delivery, pending, reverted, contactless.
- TF-IDF jako punkt odniesienia; n-gramy do porównania na walidacji.
- *(dopisek asystenta)* Normalizacja `\w+` rozbija skróty z apostrofem („hasn't” → „hasn” + „t”), stąd „hasn” wśród najczęstszych słów. Domyślny wzorzec tokenów `TfidfVectorizer` (`(?u)\w\w+`) robi to samo — do decyzji w N1: wzorzec tokenów albo n-gramy znakowe.

## 8. Dane osobowe (PII) — kontrola na train

Audyt z 30.09 (5 wzorców: e-mail, URL, ≥ 9 cyfr, IBAN, przedstawianie się) dał 0 trafień — to nie dowodzi braku PII. Wykrycie istotnych PII = stop i decyzja o zbiorze. Wypisuj tylko liczby trafień i pozycje, nie teksty.

In [ ]:
import re

pii_patterns = {
    "e-mail": r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b",
    "URL": r"\b(?:https?://|www\.)\S+",
    "co najmniej 9 cyfr": r"\d(?:[\s().+-]*\d){8,}",
    "IBAN": r"\b[A-Z]{2}\d{2}(?: ?[A-Z0-9]){11,30}\b",
    "przedstawianie się": r"\b(?:my name is|i am called|i'm called)\s+[A-Z]+\b",
}

pii_rows = []
for name, pattern in pii_patterns.items():
    mask = train["text"].str.contains(pattern, flags=re.IGNORECASE, regex=True)
    pii_rows.append({
        "wzorzec": name,
        "liczba trafień": int(mask.sum()),
        "pozycje wierszy": train.index[mask].tolist(),
    })

pii_report = pd.DataFrame(pii_rows)
display(pii_report)
print("Łączna liczba trafień wzorców:", pii_report["liczba trafień"].sum())


**Wniosek (Marcin):**
- Zero trafień pięciu wzorców na train.
- Ograniczony skan regex; brak trafień bez gwarancji braku danych osobowych.


## 9. Podsumowanie EDA (do README)

3–5 zdań: co wiemy o danych i co z tego wynika dla protokołu i modeli.

**Wnioski (Marcin):**
- 10 003 wiadomości, 77 intencji; brak pustych wartości.
- Nierówne klasy; macro-F1, wyniki per klasa i porównanie class_weight na walidacji.
- 31 par duplikatów i jeden konflikt etykiet; wspólne grupy w splicie, etykiety bez zmian.
- Krótkie teksty i podobne intencje; TF-IDF jako punkt odniesienia. *(Dopisek asystenta, 12.10.2026)* Maksimum 98 tokenów DistilBERT z tokenami specjalnymi na train; zgodnie z decyzją Marcina `max_len` zostaje 128 jak w configu.
- Zero trafień wzorców PII; ograniczony zakres kontroli.
